[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# A Searchable Archive &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell writes the logbook to `scratch/logbook`, and builds `scratch/archive.db` from it at
version 2, with the notebook's `transaction`, `migrate`, `open_archive`, `load`, `search` and
`show_plan`. Run it first. The tasks do not depend on one another, and the last cell closes the
connection and removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
from contextlib import contextmanager
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
FOLDER = SCRATCH / "logbook"
ARCHIVE = SCRATCH / "archive.db"
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
EVENTS = [
    "Heater on the sensor mast checked and working.",
    "Battery replaced after a low voltage warning.",
    "Snow cleared from the rain gauge.",
    "Sensor recalibrated against the reference thermometer.",
    "Ice on the anemometer, so the wind readings for the morning are unreliable.",
    "Annual service of the station completed.",
    "Fence repaired after a storm.",
    "Data logger restarted after a power cut, and no readings were lost.",
    "Heaters on the mast replaced.",
    "Visited twice to check the heater.",
]
ON_WARM_DAYS = {EVENTS[2]: "Grass cut around the rain gauge.", EVENTS[4]: "Anemometer bearings greased."}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


def logbook():
    """A technician's note for every station and day of 2025, written from that day's readings."""
    days = {}
    for station, hour, celsius in year_of_readings():
        days.setdefault((hour[:10], station), []).append(celsius)
    for (day, station), temperatures in sorted(days.items()):
        known = [celsius for celsius in temperatures if celsius is not None]
        if not known:
            yield station, day, "Data logger failed overnight, and there are no readings for the whole day."
            continue
        low, high = min(known), max(known)
        if high < 0:
            weather = f"Frost all day, between {low} and {high} degrees."
        elif low < 0:
            weather = f"Night frost down to {low} degrees, and a thaw to {high} by the afternoon."
        else:
            weather = f"Above freezing all day, between {low} and {high} degrees."
        day_of_year = datetime.strptime(day, "%Y-%m-%d").timetuple().tm_yday
        number = (day_of_year * 37 + list(STATIONS).index(station) * 101) % 23
        event = EVENTS[number] if number < len(EVENTS) else ""
        if low >= 0:
            event = ON_WARM_DAYS.get(event, event)
        yield station, day, f"{weather} {event}".strip()


def write_logbook(folder):
    """Write the logbook as a file for every station and month, folder/<station>/<month>.txt, and return how many."""
    months = {}
    for station, day, note in logbook():
        months.setdefault((station, day[:7]), []).append(f"{day}: {note}")
    for (station, month), lines in months.items():
        path = folder / station / f"{month}.txt"
        path.parent.mkdir(parents=True, exist_ok=True)
        heading = f"{station}, {datetime.strptime(month, '%Y-%m'):%B %Y}"
        path.write_text(heading + "\n\n" + "\n".join(lines) + "\n", encoding="utf-8")
    return len(months)


def as_words(text):
    """Text typed into a search box, as an FTS5 query that needs every word and treats none as an operator."""
    return " ".join('"' + word.replace('"', '""') + '"' for word in text.split())

@contextmanager
def transaction(conn):
    """Run a with block inside BEGIN IMMEDIATE and COMMIT, or ROLLBACK if anything in it raises."""
    conn.execute("BEGIN IMMEDIATE")
    try:
        yield
        conn.execute("COMMIT")
    except BaseException:
        conn.execute("ROLLBACK")
        raise


VERSION_1 = [
    """
    CREATE TABLE documents (
        id INTEGER PRIMARY KEY,
        path TEXT NOT NULL UNIQUE,
        station TEXT NOT NULL,
        month TEXT NOT NULL,
        body TEXT NOT NULL
    ) STRICT
    """,
    "CREATE INDEX documents_by_station_month ON documents (station, month)",
    "CREATE VIRTUAL TABLE documents_index USING fts5(body, content = 'documents', content_rowid = 'id')",
    """
    CREATE TRIGGER documents_after_insert AFTER INSERT ON documents BEGIN
        INSERT INTO documents_index (rowid, body) VALUES (new.id, new.body);
    END
    """,
    """
    CREATE TRIGGER documents_after_delete AFTER DELETE ON documents BEGIN
        INSERT INTO documents_index (documents_index, rowid, body) VALUES ('delete', old.id, old.body);
    END
    """,
    """
    CREATE TRIGGER documents_after_update AFTER UPDATE OF body ON documents BEGIN
        INSERT INTO documents_index (documents_index, rowid, body) VALUES ('delete', old.id, old.body);
        INSERT INTO documents_index (rowid, body) VALUES (new.id, new.body);
    END
    """,
]


def create_archive(conn):
    """Version 1: the documents, an index for lookups by station and month, and a full-text index kept in step."""
    for statement in VERSION_1:
        conn.execute(statement)


def migrate(conn, migrations):
    """Run every migration the archive has not had, each in a transaction with its number, and return the numbers."""
    version = conn.execute("PRAGMA user_version").fetchone()[0]
    applied = []
    for number in range(version + 1, len(migrations) + 1):
        with transaction(conn):
            migrations[number - 1](conn)
            conn.execute(f"PRAGMA user_version = {number}")
        applied.append(number)
    return applied


MIGRATIONS = [create_archive]


def open_archive(path):
    """A connection to the archive at path, in WAL mode, with rows by column name, brought up to date."""
    conn = sqlite3.connect(path, autocommit=True)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA journal_mode = WAL")
    migrate(conn, MIGRATIONS)
    return conn

UPSERT_DOCUMENT = """
    INSERT INTO documents (path, station, month, body) VALUES (?, ?, ?, ?)
    ON CONFLICT (path) DO UPDATE SET body = excluded.body WHERE documents.body IS NOT excluded.body
    RETURNING id
"""


def load(conn, folder):
    """Load every text file in folder's station folders, in one transaction, and return the paths whose text changed."""
    changed = []
    with transaction(conn):
        for path in sorted(folder.glob("*/*.txt")):
            name = path.relative_to(folder).as_posix()
            text = path.read_text(encoding="utf-8")
            if conn.execute(UPSERT_DOCUMENT, (name, path.parent.name, path.stem, text)).fetchall():
                changed.append(name)
    return changed

def search(conn, text, station=None, limit=3):
    """The documents that best match the words in text, optionally at one station, with a snippet of each."""
    return conn.execute("""
        SELECT documents.path, trim(replace(snippet(documents_index, 0, '[', ']', '...', 10), char(10), ' ')) AS snippet
        FROM documents_index JOIN documents ON documents.id = documents_index.rowid
        WHERE documents_index MATCH ? AND (? IS NULL OR documents.station = ?)
        ORDER BY documents_index.rank, documents.id
        LIMIT ?
    """, (as_words(text), station, station, limit)).fetchall()

def show_plan(conn, sql, parameters=()):
    """Print the plan SQLite chooses for a statement, one step to a line, indented under the step it belongs to."""
    depth = {0: -1}
    for step, parent, _, detail in conn.execute("EXPLAIN QUERY PLAN " + sql, parameters):
        depth[step] = depth[parent] + 1
        print("    " + "  " * depth[step] + detail)

def add_reviewed(conn):
    """Version 2: a mark for the documents someone has reviewed, 0 until someone has."""
    conn.execute("ALTER TABLE documents ADD COLUMN reviewed INTEGER NOT NULL DEFAULT 0")


MIGRATIONS = [create_archive, add_reviewed]


write_logbook(FOLDER)
conn = open_archive(ARCHIVE)
print("loaded:", len(load(conn, FOLDER)), "documents | version:", conn.execute("PRAGMA user_version").fetchone()[0])


loaded: 48 documents | version: 2


**1.** Ice on the anemometer at Tromso.


In [2]:
for row in search(conn, "ice on the anemometer", station="Tromso"):
    print(row["path"], row["snippet"])


Tromso/2025-10.txt ...[Ice] [on] [the] [anemometer], so [the] wind readings for [the]...
Tromso/2025-04.txt ...[Ice] [on] [the] [anemometer], so [the] wind readings for [the]...
Tromso/2025-01.txt ...[Ice] [on] [the] [anemometer], so [the] wind readings for [the]...


`as_words` makes every word a requirement, so a document had to contain `ice`, `on`, `the` and
`anemometer`, and `bm25` ranked first the documents that use those words most often for their length.


**2.** Documents about cutting the grass, by station.


In [3]:
for row in conn.execute("""
    SELECT documents.station, COUNT(*) AS mentions
    FROM documents_index JOIN documents ON documents.id = documents_index.rowid
    WHERE documents_index MATCH 'grass'
    GROUP BY documents.station
    ORDER BY documents.station
"""):
    print(row["station"], row["mentions"])


Bergen 9
Oslo 8
Svalbard 2
Tromso 6


`MATCH` narrows the full-text index to the documents with the word, the join brings in their
stations, and `GROUP BY` counts them as it counts any other rows. The grass is cut only on a day that
stays above freezing, so Svalbard, the coldest station, mentions it in two months. A document counts
once, however many times it mentions the grass.


**3.** Removing the documents whose files are gone.


In [4]:
def remove_missing(conn, folder):
    """Delete the documents whose files are no longer in folder, in one transaction, and return their paths."""
    removed = []
    with transaction(conn):
        for row in conn.execute("SELECT id, path FROM documents ORDER BY path").fetchall():
            if not (folder / row["path"]).exists():
                conn.execute("DELETE FROM documents WHERE id = ?", (row["id"],))
                removed.append(row["path"])
    return removed


print("before:", [row["path"] for row in search(conn, "logger failed")])
(FOLDER / "Svalbard" / "2025-03.txt").unlink()
print("removed:", remove_missing(conn, FOLDER))
print("after: ", [row["path"] for row in search(conn, "logger failed")])


before: ['Svalbard/2025-03.txt']
removed: ['Svalbard/2025-03.txt']
after:  []


The delete trigger gave FTS5 the removed document's text, so its words left the index with it, and
the search found nothing. `fetchall()` reads every path before the first `DELETE`, so the loop never
reads a table it is changing. A path stored with `/` joins onto a `Path` on any system.


**4.** Version 3, an index for reviews.


In [5]:
def index_reviews(conn):
    """Version 3: an index for finding a station's documents that nobody has reviewed."""
    conn.execute("CREATE INDEX documents_by_station_reviewed ON documents (station, reviewed)")


print("applied:", migrate(conn, [create_archive, add_reviewed, index_reviews]))
show_plan(conn, "SELECT path FROM documents WHERE station = ? AND reviewed = 0", ("Oslo",))


applied: [3]
    SEARCH documents USING INDEX documents_by_station_reviewed (station=? AND reviewed=?)


`migrate` found the archive at version 2 and ran only the third migration. The plan searches the new
index with both columns, where the index on station and month could only have narrowed the search
to the station and then read every one of its documents.


**5.** A loader that takes a mark away from a changed document.


In [6]:
UPSERT_AND_UNMARK = """
    INSERT INTO documents (path, station, month, body) VALUES (?, ?, ?, ?)
    ON CONFLICT (path) DO UPDATE SET body = excluded.body, reviewed = 0 WHERE documents.body IS NOT excluded.body
    RETURNING id
"""


def load_and_unmark(conn, folder):
    """load, with a changed document's review mark set back to 0."""
    changed = []
    with transaction(conn):
        for path in sorted(folder.glob("*/*.txt")):
            name = path.relative_to(folder).as_posix()
            text = path.read_text(encoding="utf-8")
            if conn.execute(UPSERT_AND_UNMARK, (name, path.parent.name, path.stem, text)).fetchall():
                changed.append(name)
    return changed


with transaction(conn):
    conn.execute("UPDATE documents SET reviewed = 1 WHERE station = 'Oslo'")
with (FOLDER / "Oslo" / "2025-05.txt").open("a", encoding="utf-8") as file:
    file.write("2025-05-31: Sensor recalibrated again after a storm.\n")

print("changed:", load_and_unmark(conn, FOLDER))
for row in conn.execute("SELECT reviewed, COUNT(*) AS documents FROM documents WHERE station = 'Oslo' GROUP BY reviewed"):
    print("reviewed =", row["reviewed"], "for", row["documents"], "of Oslo's documents")


changed: ['Oslo/2025-05.txt']
reviewed = 0 for 1 of Oslo's documents
reviewed = 1 for 11 of Oslo's documents


The `SET` list decides which columns a changed document gets back, so `reviewed = 0` there takes the
mark from May alone. The `WHERE` still skips every document whose text is the same, so the other
eleven months kept their marks.


**6.** A backup in memory, searched.


In [7]:
in_memory = sqlite3.connect(":memory:")
conn.backup(in_memory)
in_memory.row_factory = sqlite3.Row
for row in search(in_memory, "battery replaced"):
    print(row["path"], row["snippet"])
in_memory.close()


Svalbard/2025-11.txt ...[Battery] [replaced] after a low voltage warning. 2025-11-03...
Svalbard/2025-01.txt ...[Battery] [replaced] after a low voltage warning. 2025-01-08...
Tromso/2025-01.txt ...[Battery] [replaced] after a low voltage warning. 2025-01-09...


`backup` copied every page, the full-text index's among them, so the copy in memory searched as the
archive does, with nothing written to disk. `conn` had no transaction open, which `backup` needs from
the connection it copies.

Last, close the connection and remove the scratch folder:


In [8]:
conn.close()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [A Searchable Archive](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/19-a-searchable-archive.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
